In [ ]:
# Clean install (forces removal of any stale versions)
!pip uninstall -y transformers
!pip install transformers==4.44.0 datasets==2.21.0 evaluate==0.4.3 accelerate==0.33.0

import os
os.kill(os.getpid(), 9)  #  Forces Colab to restart runtime automatically


Found existing installation: transformers 4.56.2
Uninstalling transformers-4.56.2:
  Successfully uninstalled transformers-4.56.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 62.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 315.1/315.1 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 64.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 99.5 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attemp

In [1]:
import transformers
print(transformers.__version__)


4.44.0


In [2]:
from transformers import TrainingArguments
help(TrainingArguments)


Help on class TrainingArguments in module transformers.training_args:

class TrainingArguments(builtins.object)
 |  TrainingArguments(output_dir: str, overwrite_output_dir: bool = False, do_train: bool = False, do_eval: bool = False, do_predict: bool = False, eval_strategy: Union[transformers.trainer_utils.IntervalStrategy, str] = 'no', prediction_loss_only: bool = False, per_device_train_batch_size: int = 8, per_device_eval_batch_size: int = 8, per_gpu_train_batch_size: Optional[int] = None, per_gpu_eval_batch_size: Optional[int] = None, gradient_accumulation_steps: int = 1, eval_accumulation_steps: Optional[int] = None, eval_delay: Optional[float] = 0, torch_empty_cache_steps: Optional[int] = None, learning_rate: float = 5e-05, weight_decay: float = 0.0, adam_beta1: float = 0.9, adam_beta2: float = 0.999, adam_epsilon: float = 1e-08, max_grad_norm: float = 1.0, num_train_epochs: float = 3.0, max_steps: int = -1, lr_scheduler_type: Union[transformers.trainer_utils.SchedulerType, str] 

In [4]:
# Quick sanity checks: GPU and library versions
import torch, transformers
print("torch:", torch.__version__, "cuda available:", torch.cuda.is_available())
print("transformers:", transformers.__version__)


torch: 2.8.0+cu126 cuda available: False
transformers: 4.44.0


In [5]:
# Core libs
import numpy as np
from collections import Counter

# HF / dataset / metrics
from datasets import load_dataset, concatenate_datasets
import evaluate
from sklearn.metrics import classification_report

# Transformers
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding
)


In [6]:
# Load the easy-to-use multilingual toxicity dataset
raw = load_dataset("textdetox/multilingual_toxicity_dataset")
print("Available language splits:", list(raw.keys())[:20])  # shows language codes

# choose languages you want to train on (example: English + Hindi)
langs = ["en", "hi"]
available = [l for l in langs if l in raw]
print("Using languages:", available)

if not available:
    raise SystemExit("No chosen languages found in dataset. Run `print(raw.keys())` to see options.")

combined = concatenate_datasets([raw[l] for l in available]).shuffle(seed=42)
print("Total combined rows:", len(combined))
print("Columns:", combined.column_names)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating en split:   0%|          | 0/5000 [00:00<?, ? examples/s]

Generating ru split:   0%|          | 0/5000 [00:00<?, ? examples/s]

Generating uk split:   0%|          | 0/5000 [00:00<?, ? examples/s]

Generating de split:   0%|          | 0/5000 [00:00<?, ? examples/s]

...

Generating he split:   0%|          | 0/2011 [00:00<?, ? examples/s]

Generating hin split:   0%|          | 0/4363 [00:00<?, ? examples/s]

Generating tt split:   0%|          | 0/5000 [00:00<?, ? examples/s]

Generating ja split:   0%|          | 0/5000 [00:00<?, ? examples/s]

Available language splits: ['en', 'ru', 'uk', 'de', 'es', 'am', 'zh', 'ar', 'hi', 'it', 'fr', 'he', 'hin', 'tt', 'ja']
Using languages: ['en', 'hi']
Total combined rows: 10000
Columns: ['text', 'toxic']


In [7]:
# find text and label columns robustly
cols = combined.column_names
print("columns:", cols)

text_col = "text" if "text" in cols else cols[0]
# typical label column name in this dataset: "toxic"
if "toxic" in cols:
    label_col = "toxic"
elif "label" in cols:
    label_col = "label"
else:
    # fallback - pick any numeric-looking column except text
    label_col = next(c for c in cols if c != text_col)

print("Text column:", text_col)
print("Label column:", label_col)

# show distribution quickly
print("Label distribution (full):")
from collections import Counter
print(Counter(combined[label_col]))


columns: ['text', 'toxic']
Text column: text
Label column: toxic
Label distribution (full):
Counter({0: 5000, 1: 5000})


In [8]:
# For development, keep a small subset to iterate quickly
max_samples = min(4000, len(combined))   # change or remove for full training
ds_small = combined.select(range(max_samples))
split = ds_small.train_test_split(test_size=0.2, seed=42)

train_ds = split["train"]
test_ds  = split["test"]

print("Train size:", len(train_ds), "Test size:", len(test_ds))


Train size: 3200 Test size: 800


In [9]:
MODEL_NAME = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(batch):
    return tokenizer(batch[text_col], truncation=True, max_length=128)

# Use batched mapping; remove text to keep only tokenized inputs + label
train_tok = train_ds.map(tokenize_fn, batched=True, remove_columns=[text_col])
test_tok  = test_ds.map(tokenize_fn, batched=True, remove_columns=[text_col])

# rename label column to 'labels' (Trainer expects that)
train_tok = train_tok.rename_column(label_col, "labels")
test_tok  = test_tok.rename_column(label_col, "labels")

# convert to pytorch tensors format (optional but useful)
train_tok.set_format("torch")
test_tok.set_format("torch")

print(train_tok.features)

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Map:   0%|          | 0/3200 [00:00<?, ? examples/s]

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

{'labels': Value(dtype='int64', id=None), 'input_ids': Sequence(feature=Value(dtype='int32', id=None), length=-1, id=None), 'attention_mask': Sequence(feature=Value(dtype='int8', id=None), length=-1, id=None)}


In [10]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1_macro": f1.compute(predictions=preds, references=labels, average="macro")["f1"]
    }

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [11]:
import os
os.environ["WANDB_DISABLED"] = "true"


In [12]:
# use fp16 automatically if CUDA available (speeds training on GPU)
use_fp16 = True if torch.cuda.is_available() else False

training_args = TrainingArguments(
    output_dir="./xlmr_toxic_small",
    evaluation_strategy="epoch",   # run evaluation at end of each epoch
    save_strategy="no",            # don't save intermediate checkpoints for this demo
    per_device_train_batch_size=8, # reduce this if you run out of memory
    per_device_eval_batch_size=8,
    num_train_epochs=1,            # 1 epoch for quick demo; increase for production
    logging_steps=20,
    learning_rate=2e-5,
    load_best_model_at_end=False,
    fp16=use_fp16
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=test_tok,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)



/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [13]:
# 1) Train the model
train_result = trainer.train()
print("Train finished. Check training metrics:", train_result)

# 2) Evaluate on test set & show sklearn report
pred_out = trainer.predict(test_tok)   # runs eval on test_tok
preds = np.argmax(pred_out.predictions, axis=-1)
labels = pred_out.label_ids

print("Metrics (custom):", compute_metrics((pred_out.predictions, labels)))
from sklearn.metrics import classification_report
print("\nClassification report:\n", classification_report(labels, preds, digits=4))

# 3) Save model + tokenizer (this will include any updated config)
trainer.save_model("xlmr_toxic_finetuned")
tokenizer.save_pretrained("xlmr_toxic_finetuned")


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.428000,0.276017,0.935000,0.934896


Train finished. Check training metrics: TrainOutput(global_step=400, training_loss=0.3729573690891266, metrics={'train_runtime': 3757.3527, 'train_samples_per_second': 0.852, 'train_steps_per_second': 0.106, 'total_flos': 109688532313440.0, 'train_loss': 0.3729573690891266, 'epoch': 1.0})


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Metrics (custom): {'accuracy': 0.935, 'f1_macro': 0.9348958333333333}

Classification report:
               precision    recall  f1-score   support

           0     0.9489    0.9264    0.9375       421
           1     0.9203    0.9446    0.9323       379

    accuracy                         0.9350       800
   macro avg     0.9346    0.9355    0.9349       800
weighted avg     0.9354    0.9350    0.9350       800



('xlmr_toxic_finetuned/tokenizer_config.json',
 'xlmr_toxic_finetuned/special_tokens_map.json',
 'xlmr_toxic_finetuned/sentencepiece.bpe.model',
 'xlmr_toxic_finetuned/added_tokens.json',
 'xlmr_toxic_finetuned/tokenizer.json')

In [14]:
from transformers import pipeline

device = 0 if torch.cuda.is_available() else -1

# Create pipeline
clf = pipeline("text-classification", model=trainer.model, tokenizer=tokenizer, device=device)

# Custom test examples
examples = [
    "You are worthless and disgusting!",
    "We should all live in harmony.",
    "Go back to your country!",
    "This is a beautiful piece of work!"
]

# Run predictions
for text in examples:
    print(text, "->", clf(text))


You are worthless and disgusting! -> [{'label': 'LABEL_1', 'score': 0.994572103023529}]
We should all live in harmony. -> [{'label': 'LABEL_0', 'score': 0.9929094910621643}]
Go back to your country! -> [{'label': 'LABEL_0', 'score': 0.992149829864502}]
This is a beautiful piece of work! -> [{'label': 'LABEL_0', 'score': 0.9962167143821716}]


In [16]:
!zip -r xlmr_toxic_finetuned.zip xlmr_toxic_finetuned
from google.colab import files
files.download("xlmr_toxic_finetuned.zip")


updating: xlmr_toxic_finetuned/ (stored 0%)
updating: xlmr_toxic_finetuned/sentencepiece.bpe.model (deflated 49%)
updating: xlmr_toxic_finetuned/tokenizer.json (deflated 76%)
updating: xlmr_toxic_finetuned/tokenizer_config.json (deflated 77%)
updating: xlmr_toxic_finetuned/training_args.bin (deflated 53%)
updating: xlmr_toxic_finetuned/special_tokens_map.json (deflated 52%)
updating: xlmr_toxic_finetuned/config.json (deflated 51%)
updating: xlmr_toxic_finetuned/model.safetensors


zip error: Interrupted (aborting)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [17]:
import gradio as gr

def predict(text):
    result = clf(text)[0]
    return f"{text} ➜ {result['label']} (score: {result['score']:.2f})"

gr.Interface(fn=predict, inputs="text", outputs="text", title="Hate Speech Detector").launch()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://53c70d8c72b41e1f67.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
